# NB4 — Negative V1 → scorer-ready dataset → final freeze

Notebook này hoàn tất data processing V1 sau khi NB3 đã PASS:

1. sinh **1 synthetic negative / 1 final positive**;
2. replacement cùng `master_category`, khác item, khác kit và cùng official split;
3. ghép từng positive với negative thành scorer-ready JSONL;
4. validator tự dựng lại từng cặp để kiểm tra đúng một swap và toàn bộ provenance;
5. kiểm tra label balance, metadata, embedding gate và cross-split leakage;
6. tạo SHA-256 manifests và chỉ trả `READY_TO_TRAIN` khi mọi gate đều pass.

> Negative V1 vẫn là random synthetic negative, không phải hard negative.

## 1. Runtime portable

Notebook không tự mount Drive hoặc tự checkout/pull Git. Hãy mở từ repository đã clone. Artifact có thể nằm trên Drive hoặc local miễn path config trỏ đúng vị trí.

In [ ]:
# Không có setup bắt buộc dành riêng cho Colab.
# Xem README: FASHION_PROJECT_ROOT và FASHION_ARTIFACT_ROOT.

## 2. Tìm repository và ghi nhận commit đang chạy

Notebook dùng đúng working tree hiện tại. Nó không tự đổi branch, pull hoặc ghi đè code của người chạy.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
AUTO_CLONE_REPO = False  # Chỉ bật khi runtime chưa có repository, ví dụ Colab mới.


def find_repo_root(start: Path = Path.cwd()):
    explicit = os.environ.get("FASHION_PROJECT_ROOT")
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
        raise FileNotFoundError(f"FASHION_PROJECT_ROOT không hợp lệ: {candidate}")

    current = start.expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None and AUTO_CLONE_REPO:
    REPO_ROOT = (Path.cwd() / "opisoverated").resolve()
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Không tìm thấy repository. Hãy mở notebook từ repo đã clone, "
        "set FASHION_PROJECT_ROOT, hoặc bật AUTO_CLONE_REPO=True."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.runtime_paths import load_runtime_paths

RUNTIME_PATHS = load_runtime_paths(repo_root=REPO_ROOT)
print("Repo root      :", REPO_ROOT)
print("Path config    :", RUNTIME_PATHS.config_path)
print("Artifact root  :", RUNTIME_PATHS.artifact_root)

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
        text=True,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    GIT_COMMIT = None

print("Git commit:", GIT_COMMIT or "unavailable")

## 3. Khai báo input/output

NB4 đọc `core7_dir` và ghi `scorer_ready_dir` theo runtime path config. Input và output vẫn là hai folder riêng.

In [ ]:
CORE7_DIR = RUNTIME_PATHS.core7_dir
OUTPUT_DIR = RUNTIME_PATHS.scorer_ready_dir
EMBEDDING_REPORT = CORE7_DIR / "core7_embedding_validation_report.json"
SEED = 42
ALLOW_OVERWRITE = False

required_paths = [EMBEDDING_REPORT]
for split in ("train", "valid", "test"):
    required_paths.extend([
        CORE7_DIR / f"category_clean_{split}.jsonl",
        CORE7_DIR / f"core7_item_metadata_v1_{split}.jsonl",
    ])

for path in required_paths:
    print(path.name, "exists=", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUTPUT_DIR)
print("Seed         :", SEED)

## 4. Build full scorer-ready dataset V1

`ALLOW_OVERWRITE=False` bảo vệ versioned artifacts. Nếu V1 đã tồn tại, notebook dừng thay vì âm thầm tạo benchmark khác cùng tên.

In [ ]:
from src.data.build_core7_scorer_dataset import build_scorer_dataset_v1

result = build_scorer_dataset_v1(
    data_dir=CORE7_DIR,
    output_dir=OUTPUT_DIR,
    embedding_report_path=EMBEDDING_REPORT,
    seed=SEED,
    git_commit=GIT_COMMIT,
    overwrite=ALLOW_OVERWRITE,
)

## 5. Đọc negative sampling report

In [ ]:
for split in ('train', 'valid', 'test'):
    sampling = result['sampling_reports'][split]
    merge = sampling['merge']
    print(split.upper())
    print('  positive input     :', sampling['positive_count'])
    print('  negatives generated:', sampling['negative_count'])
    print('  generation coverage:', f"{sampling['generation_coverage']:.4%}")
    print('  failed positives   :', sampling['failed_positive_count'])
    print('  sampling pass      :', sampling['pass'])
    print('  merge pass         :', merge['pass'])
    if sampling['failure_examples']:
        print('  failure examples   :', sampling['failure_examples'][:5])
    print()

## 6. Đọc final validation gate

In [ ]:
final_report = result['final_validation']
for split in ('train', 'valid', 'test'):
    split_report = final_report['splits'][split]
    print(split.upper())
    print('  samples   :', split_report['sample_count'])
    print('  positives :', split_report['positive_count'])
    print('  negatives :', split_report['negative_count'])
    print('  issues    :', split_report['issue_count'])
    print('  pass      :', split_report['pass'])
    if split_report['issue_examples']:
        print('  examples  :', split_report['issue_examples'][:5])
    print()

print('Embedding gate              :', final_report['embedding_validation_pass'])
print('Negative sampling gate      :', final_report['negative_sampling_pass'])
print('Source-kit cross-split      :', final_report['source_kit_cross_split_count'])
print('Item cross-split            :', final_report['item_cross_split_count'])
print('Global duplicate sample IDs :', final_report['global_duplicate_sample_id_count'])
print('FINAL STATUS                :', result['status'])

## 7. Expected outputs

Folder `scorer_ready_v1` sẽ chứa:

```text
negative_v1_train/valid/test.jsonl
scorer_ready_v1_train/valid/test.jsonl
negative_sampling_v1_*_report.json
final_validation_v1.json
split_manifest_v1.json
dataset_manifest_v1.json
```

Chỉ khi `FINAL STATUS = READY_TO_TRAIN` thì Type-aware Pairwise scorer mới được dùng các `scorer_ready_v1_*` để train/validation/test.